In [2]:
!pip install datasets

from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')

import pandas as pd
df = pd.DataFrame(dataset)
df = df[['title', 'abstract']]
df = df.head(15000)   # keep subset for speed
df.isnull().sum()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,0
title,0
abstract,0


In [3]:
df["paper_text"] = df["title"] + " " + df["abstract"]
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False).str.strip()

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding = model.encode(
    df["paper_text"].to_list(),
    batch_size=32,
    show_progress_bar=True
)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [5]:
!pip install faiss-cpu
import faiss

faiss.normalize_L2(embedding)
index = faiss.IndexFlatIP(384)   # 384 = MiniLM embedding dim
index.add(embedding)
print(index.ntotal)

15000


In [8]:
def search_paper(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:300])
        print("---")

search_paper("deep learning for medical image analysis")

Score: 0.6807244
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image

---
Score: 0.67092204
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract:   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large 
---
Score: 0.65219975
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection
Abstract:   The classification of MRI images according to the anatomical field of vi

In [6]:
!pip install transformers==4.46.3
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        abstract = df.iloc[idx]["abstract"]
        summary = summarizer(abstract, max_length=120, min_length=40)
        print("Title:", df.iloc[idx]["title"])
        print("Summary:", summary[0]["summary_text"])
        print("---")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [8]:
!pip install keybert==0.8.5 spacy
!python -m spacy download en_core_web_sm

from keybert import KeyBERT
import spacy

kw_model = KeyBERT(model)
nlp = spacy.load("en_core_web_sm")

def extract_keywords(text):
    return kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3), stop_words="english")

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 63.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

candidate_labels = ["Computer Vision", "Natural Language Processing", "Reinforcement Learning",
                    "Generative Models", "Optimization", "Graph Neural Networks", "Robotics"]

def classify_paper(text):
    result = classifier(text, candidate_labels)
    return result["labels"][0], result["scores"][0]

def answer_question(context, question):
    result = qa_pipeline(question=question, context=context)
    return result["answer"], result["score"]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def full_paper_analysis(query, k=5, question=None):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        abstract = df.iloc[idx]["abstract"]
        title = df.iloc[idx]["title"]
        summary = summarizer(abstract, max_length=120, min_length=40)[0]["summary_text"]
        keywords = extract_keywords(abstract)
        entities = extract_entities(summary)
        label, conf = classify_paper(abstract)

        print("Title:", title)
        print("Score:", score)
        print("Summary:", summary)
        print("Keywords:", keywords)
        print("Entities:", entities)
        print("Domain:", label, conf)

        if question:
            ans, conf = answer_question(abstract, question)
            print("Q:", question, "-> A:", ans, f"(conf {conf:.2f})")
        print("=====")

full_paper_analysis("deep learning for medical image analysis", k=5,
                    question="What method does the paper use?")

Title: A Perspective on Deep Imaging
Score: 0.68072414
Summary: The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.
Keywords: [('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]
Entities: []
Domain: Optimization 0.2972254455089569
Q: What method does the paper use? -> A: tomographic imaging and deep learning, or machine learning (conf 0.13)
=====
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Score: 0.67092216
Summary: Training a deep convolutional neural network from scratch is difficult because it requires a large amount of labeled traini

In [12]:
def recommend_similar_papers(idx, k=5):
    query_embedding = embedding[idx].reshape(1, -1)
    D, I = index.search(query_embedding, k+1)   # +1 kyunki khud bhi match karega
    print("Papers similar to:", df.iloc[idx]["title"])
    for score, i in zip(D[0], I[0]):
        if i == idx:
            continue
        print(f"- {df.iloc[i]['title']} (score: {score:.3f})")

recommend_similar_papers(10466, k=5)

Papers similar to: A Perspective on Deep Imaging
- Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection (score: 0.669)
- Spatially-Adaptive Reconstruction in Computed Tomography using Neural
  Networks (score: 0.657)
- Deep Learning for Photoacoustic Tomography from Sparse Data (score: 0.642)
- Framing U-Net via Deep Convolutional Framelets: Application to
  Sparse-view CT (score: 0.619)
- High-Resolution Breast Cancer Screening with Multi-View Deep
  Convolutional Neural Networks (score: 0.619)


In [22]:
df = pd.DataFrame(dataset)[['title', 'abstract']].head(1000)

In [17]:
!pip install pdfplumber

import pdfplumber
from google.colab import files

def upload_and_extract_pdf():
    uploaded = files.upload()  # opens file picker in Colab
    filename = list(uploaded.keys())[0]
    text = ""
    with pdfplumber.open(filename) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text.strip()

# usage
custom_paper_text = upload_and_extract_pdf()
print(custom_paper_text[:500])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 80.4 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


Saving 1.pdf to 1.pdf
Microchemical Journal 219 (2025) 116032
Contents lists available at ScienceDirect
Microchemical Journal
journal homepage: www.elsevier.com/locate/microc
Non-invasive blood glucose prediction model using machine learning with
t-test-based feature selection on saliva FTIR spectra
Jing Yina,b,c,1, Baorong Fud,1, Zhushanying Zhanga,b,c,*,1,
Ju Wanga,b,c, Huimin Caoa,b,c, Yuan Gaoa,b,c
aCollege of Biomedical Engineering, South-Central MinZu University, Wuhan 430074, China
bKey Laboratory of Cognitive
